# Module 1: Building a Single Agent with Role-Based Access Control

## Overview

In this module, you'll build a **single Product Catalog Agent** with role-based access control (RBAC) using the Strands Agent SDK and **MCP (Model Context Protocol)** tools.

### Why Start with a Single Agent?
Before building complex multi-agent systems, it's critical to:
1. **Validate tool calling** - Ensure the agent correctly selects and uses the right tools
2. **Test access control** - Verify that role-based restrictions work properly
3. **Evaluate effectiveness** - Measure a single agent's quality before adding orchestration complexity

### What You'll Build

```
┌──────────────────────────────────┐
│      User Request               │
│  (with role: customer or admin)  │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   PRODUCT CATALOG AGENT          │
│   (Claude Haiku 4.5)             │
│   • Role-aware system prompt     │
│   • Tool filtering by role       │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   Product MCP Server (FastMCP)   │
│                                  │
│  READ tools (customer + admin):  │
│  • search_products               │
│  • get_product_details           │
│  • check_inventory               │
│  • get_product_recommendations   │
│  • compare_products              │
│  • get_return_policy             │
│                                  │
│  WRITE tools (admin only):       │
│  • create_product                │
│  • update_product                │
│  • delete_product                │
│  • update_inventory              │
│  • update_pricing                │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│       DynamoDB Products Table    │
└──────────────────────────────────┘
```

### Learning Objectives
1. Build an agent with Strands SDK connected to an MCP tool server
2. Understand MCP architecture: agent ↔ MCP client ↔ MCP server ↔ DynamoDB
3. Implement role-based access control by filtering tools per user role
4. Test different personas (customer vs admin) against the same agent
5. Validate access control boundaries

## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [1]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

AWS Region: us-east-1
Products Table: ecommerce-workshop-products


## Step 2: Understanding MCP Tools

Our agent uses **MCP (Model Context Protocol)** to access tools from a separate MCP server process. This provides:
- **Decoupling**: Tools run as a separate process from the agent
- **Reusability**: MCP servers can be shared across multiple agents
- **Standardization**: Follows the MCP protocol for tool discovery and invocation

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)        Products Table
```

### Tools in Our MCP Server

| Tool | Access Level | Description |
|------|-------------|-------------|
| `search_products` | Customer + Admin | Search catalog by keywords/category |
| `get_product_details` | Customer + Admin | Get full product details |
| `check_inventory` | Customer + Admin | Check stock availability |
| `get_product_recommendations` | Customer + Admin | Get product recommendations |
| `compare_products` | Customer + Admin | Side-by-side product comparison |
| `get_return_policy` | Customer + Admin | Return policy information |
| `create_product` | **Admin only** | Add new product to catalog |
| `update_product` | **Admin only** | Modify product information |
| `delete_product` | **Admin only** | Remove product (soft delete) |
| `update_inventory` | **Admin only** | Adjust stock quantities |
| `update_pricing` | **Admin only** | Change prices, set sales |

In [2]:
# Verify MCP server exists
from pathlib import Path

mcp_server_path = Path('mcp_servers/product_mcp_server.py')
print(f"Product MCP Server: {mcp_server_path} - {'exists' if mcp_server_path.exists() else 'MISSING'}")

Product MCP Server: mcp_servers/product_mcp_server.py - exists


## Step 3: Build the Product Catalog Agent (No RBAC)

Let's first build a basic agent that connects to the MCP server with **all tools available** (no role filtering yet). This lets us verify the MCP connection and tool calling work correctly.

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Model configuration
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
print(f"Model: {HAIKU_MODEL_ID}")

# Connect to the product MCP server
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(mcp_server_path)],
    env={
        **os.environ,
        "AWS_REGION": REGION,
        "PRODUCTS_TABLE": os.environ.get('PRODUCTS_TABLE_NAME', 'ecommerce-workshop-products')
    }
)

mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()

# Discover all tools from the MCP server
all_tools = mcp_client.list_tools_sync()
print(f"\nDiscovered {len(all_tools)} tools from MCP server:")
for tool in all_tools:
    print(f"  - {tool.tool_name}")

Model: global.anthropic.claude-haiku-4-5-20251001-v1:0

Discovered 11 tools from MCP server:
  - search_products
  - get_product_details
  - check_inventory
  - get_product_recommendations
  - compare_products
  - get_return_policy
  - create_product
  - update_product
  - delete_product
  - update_inventory
  - update_pricing


In [4]:
# Create a basic agent with ALL tools (no RBAC yet)
basic_agent = Agent(
    name="ProductCatalogAgent",
    model=BedrockModel(
        model_id=HAIKU_MODEL_ID,
        region_name=REGION,
        temperature=0.3,
        max_tokens=1500
    ),
    system_prompt="You are a Product Catalog Assistant. Help users with product queries using the available tools.",
    tools=all_tools, 
    callback_handler=None # disable the console output.
)
print("Basic agent created with all 11 tools (no role filtering)")

Basic agent created with all 11 tools (no role filtering)


## Step 4: Test Basic Tool Calling

Let's verify the agent correctly uses MCP tools to answer product queries.

In [5]:
# Test 1: Product search
print("=" * 60)
print("Test 1: Product Search")
print("=" * 60)
response = basic_agent("Do you have any wireless headphones under $100?")
print(f"\nResponse: {response}")

Test 1: Product Search

Response: Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones** (PROD-001)
- **Price:** $79.99
- **In Stock:** Yes
- **Description:** Premium over-ear wireless headphones with 40-hour battery life, active noise cancellation, and comfortable memory foam ear cushions.

This is a great deal at under $100 with excellent features like active noise cancellation and an impressive 40-hour battery life!

I also found another option that's slightly above your budget:
- **Noise Canceling Earbuds** (PROD-055) - $149.99 - True wireless earbuds with adaptive noise cancellation and 8-hour battery life

Would you like more details about the $79.99 headphones, or would you like me to help you with anything else?



In [6]:
# Test 2: Product details
print("=" * 60)
print("Test 2: Product Details")
print("=" * 60)
response = basic_agent("Tell me about the Gaming Mechanical Keyboard PROD-077")
print(f"\nResponse: {response}")

Test 2: Product Details

Response: Here are the details for the **Gaming Mechanical Keyboard RGB** (PROD-077):

**Price:** $159.99

**Description:** Full-size mechanical gaming keyboard with hot-swappable switches, per-key RGB lighting, and detachable wrist rest.

**Key Specifications:**
- **Switch Type:** Hot-swappable (Cherry MX compatible) - allows you to customize and replace switches
- **Lighting:** Per-key RGB with 16.8M colors for full customization
- **Connection:** USB-C detachable cable for easy portability
- **Anti-Ghosting:** Full N-key rollover for responsive gaming
- **Media Controls:** Dedicated media keys + volume wheel

**Availability:** In Stock (60 units available)

**Rating:** 4.6 out of 5 stars

**Warranty:** 2 years

**Return Policy:** 30-day return policy

This is a solid gaming keyboard with excellent features for gamers, including hot-swappable switches for customization and per-key RGB lighting. The full N-key rollover ensures no missed inputs during intense g

In [7]:
# Test 3: Inventory check
print("=" * 60)
print("Test 3: Inventory Check")
print("=" * 60)
response = basic_agent("Is the 4K webcam PROD-088 in stock?")
print(f"\nResponse: {response}")

Test 3: Inventory Check

Response: Unfortunately, the **4K Webcam (PROD-088) is currently out of stock**. 

However, the good news is that it's expected to be restocked on **January 15, 2025**.

Would you like me to:
1. Get more details about this 4K webcam?
2. Search for alternative webcams that are currently in stock?
3. Help you with anything else?



In [8]:
# Test 4: Product comparison
print("=" * 60)
print("Test 4: Product Comparison")
print("=" * 60)
response = basic_agent("Compare the Wireless Bluetooth Headphones PROD-001 with the Noise Canceling Earbuds PROD-055")
print(f"\nResponse: {response}")

Test 4: Product Comparison

Response: Here's a side-by-side comparison of the two audio products:

| Feature | PROD-001: Wireless Bluetooth Headphones | PROD-055: Noise Canceling Earbuds |
|---------|----------------------------------------|-----------------------------------|
| **Price** | $79.99 | $149.99 |
| **Type** | Over-ear headphones | True wireless earbuds |
| **Rating** | 4.5/5 | 4.5/5 |
| **In Stock** | Yes | Yes |
| **Battery Life** | 40 hours | 8 hours (earbuds) + 24 hours (with case) |
| **Noise Cancellation** | Active ANC | Adaptive noise cancellation |
| **Bluetooth Version** | 5.3 | 5.2 |
| **Driver Size** | 40mm | N/A |
| **Weight** | 250g | N/A |
| **Water Resistance** | N/A | IPX4 |
| **Codecs** | N/A | AAC, aptX Adaptive |
| **Warranty** | 1 year | 1 year |

**Key Differences:**

- **PROD-001** is the **budget-friendly option** at $79.99 with exceptional 40-hour battery life, making it ideal for extended use without charging
- **PROD-055** is the **premium option**

In [9]:
# Clean up the basic agent's MCP connection
mcp_client.__exit__(None, None, None)
print("Basic agent MCP connection cleaned up")

Basic agent MCP connection cleaned up


## Step 5: Understanding Role-Based Access Control (RBAC)

Now that we've confirmed the agent and tools work, let's add **access control**.

### The Problem
Our MCP server exposes 11 tools - 6 read-only tools (safe for customers) and 5 write tools (admin-only). Without RBAC, any user could create, update, or delete products.

### The Solution: Tool Filtering

We implement RBAC by **filtering which tools the agent receives** based on the user's role:

```python
# Customer gets 6 read-only tools
customer_tools = filter_by_role(all_tools, role="customer")  # 6 tools

# Admin gets all 11 tools
admin_tools = filter_by_role(all_tools, role="admin")  # 11 tools
```

Since the LLM can only call tools it knows about, **removing a tool from the agent's tool list is an effective access control boundary**.

### Production Mapping (Module 03)
In production, user roles come from **AgentCore Identity** via JWT tokens:
- A **Cognito User Pool** manages users with `customer` and `admin` groups
- The **JWT inbound authorizer** on AgentCore Runtime validates tokens
- Agent code extracts the role from JWT claims and filters tools accordingly

In [10]:
# Add agents to path and import the RBAC-enabled agent
%load_ext autoreload
%autoreload 2

sys.path.insert(0, 'agents')
from product_catalog_agent import (
    ProductCatalogAgent,
    UserSession,
    create_product_catalog_agent,
    CUSTOMER_TOOLS,
    ADMIN_TOOLS,
    ADMIN_ONLY_TOOLS,
    CUSTOMER_PERSONAS,
    ADMIN_PERSONAS,
)

print("RBAC Configuration:")
print(f"  Customer tools ({len(CUSTOMER_TOOLS)}): {CUSTOMER_TOOLS}")
print(f"  Admin-only tools ({len(ADMIN_ONLY_TOOLS)}): {ADMIN_ONLY_TOOLS}")
print(f"  Total admin tools ({len(ADMIN_TOOLS)}): {len(ADMIN_TOOLS)}")

RBAC Configuration:
  Customer tools (6): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy']
  Admin-only tools (5): ['create_product', 'update_product', 'delete_product', 'update_inventory', 'update_pricing']
  Total admin tools (11): 11


## Step 6: Create User Sessions (Personas)

Let's define our test personas. In production, these would come from JWT claims.

In [11]:
# Define test personas
customer_session = UserSession(
    user_id="CUST-1001",
    role="customer",
    email="john.smith@email.com",
    name="John Smith"
)

admin_session = UserSession(
    user_id="ADMIN-001",
    role="admin",
    email="alice.admin@company.com",
    name="Alice (Admin)"
)

print("Test Personas:")
print(f"  Customer: {customer_session.name} ({customer_session.email}) - role: {customer_session.role}")
print(f"  Admin:    {admin_session.name} ({admin_session.email}) - role: {admin_session.role}")

Test Personas:
  Customer: John Smith (john.smith@email.com) - role: customer
  Admin:    Alice (Admin) (alice.admin@company.com) - role: admin


## Step 7: Test Customer Persona

The customer agent should be able to search, view, and compare products, but should **NOT** be able to create, update, or delete products.

In [12]:
# Create agent with CUSTOMER role
customer_agent = create_product_catalog_agent(
    region=REGION,
    user_session=customer_session
)

user_info = customer_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

Agent created for: John Smith (role: customer)
Available tools (6): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy']


In [13]:
# Customer Test 1: Search (should work)
print("=" * 60)
print("Customer Test 1: Product Search (ALLOWED)")
print("=" * 60)
response = customer_agent("I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nResponse: {response}")

Customer Test 1: Product Search (ALLOWED)
I'll search for gaming keyboards with RGB lighting for you.
Tool #1: search_products
Great! I found a gaming keyboard with RGB lighting. Let me get you more detailed information about it.
Tool #2: get_product_details

Tool #3: check_inventory
Perfect! Here's what I found for you:

## **Gaming Mechanical Keyboard RGB** (PROD-077)
**Price:** $159.99

### Key Features:
- **RGB Lighting:** Per-key RGB with 16.8M colors - fully customizable
- **Switches:** Hot-swappable Cherry MX compatible switches (easy to replace)
- **Layout:** Full-size mechanical keyboard
- **Connection:** USB-C detachable cable for portability
- **Anti-Ghosting:** Full N-key rollover for competitive gaming
- **Extras:** Dedicated media keys + volume wheel, detachable wrist rest

### Availability & Support:
- ✅ **In Stock:** 60 units available, ready to ship
- ⭐ **Rating:** 4.6 out of 5 stars
- 🛡️ **Warranty:** 2 years
- 📋 **Return Policy:** 30-day return policy

This is a soli

In [14]:
# Customer Test 2: Recommendations (should work)
print("=" * 60)
print("Customer Test 2: Recommendations (ALLOWED)")
print("=" * 60)
response = customer_agent("Can you recommend some audio products for a music lover?")
print(f"\nResponse: {response}")

Customer Test 2: Recommendations (ALLOWED)
I'll get some personalized audio product recommendations for you based on your interest in music.
Tool #4: get_product_recommendations
Excellent! Here are some great audio products for a music lover:

## Recommended Audio Products:

### 1. **Wireless Bluetooth Headphones** (PROD-001)
**Price:** $79.99
- Premium over-ear design with memory foam ear cushions
- 40-hour battery life (excellent for extended listening)
- Active noise cancellation
- Great value for the price!

### 2. **Noise Canceling Earbuds** (PROD-055)
**Price:** $149.99
- True wireless with adaptive noise cancellation
- Transparency mode for ambient awareness
- 8-hour battery per charge
- Perfect for on-the-go listening

### 3. **Portable Bluetooth Speaker** (PROD-033)
**Price:** $199.99
- 360-degree sound for immersive listening
- Waterproof design (great for outdoor use)
- 20-hour battery life
- Party mode for connecting multiple speakers

### 4. **Premium Gaming Headset** (PRO

In [15]:
# Customer Test 3: Attempt admin action (should be REFUSED)
print("=" * 60)
print("Customer Test 3: Create Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Create a new product called 'Super Speaker' with price $299.99 in the Audio category")
print(f"\nResponse: {response}")

Customer Test 3: Create Product (SHOULD BE REFUSED)
Sorry that I couldn't do so!

I'm not able to create, update, delete, or modify products in the catalog. I can only help you search for existing products, view their details, check inventory, get recommendations, and compare products.

If you need to add a new product to the catalog, you'll need to contact your store administrator or use the product management system directly.

Is there anything else I can help you with regarding existing products? I'd be happy to search for speakers, compare audio products, or help you find something specific!
Response: Sorry that I couldn't do so!

I'm not able to create, update, delete, or modify products in the catalog. I can only help you search for existing products, view their details, check inventory, get recommendations, and compare products.

If you need to add a new product to the catalog, you'll need to contact your store administrator or use the product management system directly.

Is the

In [16]:
# Customer Test 4: Attempt to delete a product (should be REFUSED)
print("=" * 60)
print("Customer Test 4: Delete Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Delete the product PROD-001 from the catalog")
print(f"\nResponse: {response}")

Customer Test 4: Delete Product (SHOULD BE REFUSED)
Sorry that I couldn't do so!

I'm not able to delete, create, update, or modify products in the catalog. I can only help you search for existing products, view their details, check inventory, get recommendations, and compare products.

If you need to remove a product from the catalog, you'll need to contact your store administrator or use the product management system directly.

Is there anything else I can help you with? I'm happy to assist with searching, comparing, or getting information about existing products!
Response: Sorry that I couldn't do so!

I'm not able to delete, create, update, or modify products in the catalog. I can only help you search for existing products, view their details, check inventory, get recommendations, and compare products.

If you need to remove a product from the catalog, you'll need to contact your store administrator or use the product management system directly.

Is there anything else I can help y

In [17]:
# Clean up customer agent
customer_agent.cleanup()
print("Customer agent cleaned up")

Customer agent cleaned up


## Step 8: Test Admin Persona

The admin agent should have **full access** to all tools, including creating, updating, and deleting products.

In [28]:
# Create agent with ADMIN role
admin_agent = create_product_catalog_agent(
    region=REGION,
    user_session=admin_session
)

user_info = admin_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

Agent created for: Alice (Admin) (role: admin)
Available tools (11): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy', 'create_product', 'update_product', 'delete_product', 'update_inventory', 'update_pricing']


In [29]:
# Admin Test 1: Search still works (admin has all customer tools too)
print("=" * 60)
print("Admin Test 1: Product Search (ALLOWED)")
print("=" * 60)
# Search for all audio products with noise cancelling
response = admin_agent("Search for all audio products")
print(f"\nResponse: {response}")

Admin Test 1: Product Search (ALLOWED)



Response: Perfect! I found **4 audio products** in the catalog. Here's a summary:

| Product ID | Product Name | Price | Stock Status |
|---|---|---|---|
| **PROD-001** | Wireless Bluetooth Headphones | $79.99 | ✅ In Stock |
| **PROD-055** | Noise Canceling Earbuds | $149.99 | ✅ In Stock |
| **PROD-200** | Premium Gaming Headset | $129.99 | ✅ In Stock |
| **PROD-033** | Portable Bluetooth Speaker | $199.99 | ✅ In Stock |

**Product Details:**

1. **PROD-001 - Wireless Bluetooth Headphones** ($79.99)
   - Premium over-ear wireless headphones with 40-hour battery life, active noise cancellation, and comfortable memory foam ear cushions.

2. **PROD-055 - Noise Canceling Earbuds** ($149.99)
   - True wireless earbuds with adaptive noise cancellation, transparency mode, and 8-hour battery life per charge.

3. **PROD-200 - Premium Gaming Headset** ($129.99)
   - Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.

4. **PROD-033 - Portable Bluetooth 

In [30]:
# Admin Test 2: Create a new product
print("=" * 60)
print("Admin Test 2: Create Product (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Create a new product with ID PROD-200 called 'Premium Gaming Headset' "
    "in the Audio category for $129.99. Description: 'Professional gaming headset "
    "with 7.1 surround sound, detachable microphone, and RGB lighting.' "
    "Specifications: bluetooth 5.3, weight 320g, driver size 50mm. "
    "Set initial stock to 50 units."
)
print(f"\nResponse: {response}")

Admin Test 2: Create Product (ALLOWED)

Response: I notice that **PROD-200 already exists** in the catalog with the exact same name, category, price, and description you're trying to create. 

The existing product is:
- **Product ID:** PROD-200
- **Name:** Premium Gaming Headset
- **Category:** Audio
- **Price:** $129.99
- **Description:** Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.

**Would you like me to:**

1. **Update the existing PROD-200** with the new specifications (bluetooth 5.3, weight 320g, driver size 50mm) and adjust the stock to 50 units?

2. **Create a new product with a different ID** (e.g., PROD-201, PROD-202, etc.) if this is meant to be a separate variant?

3. **Check the current details** of PROD-200 to see what specifications and stock it currently has?

Please let me know how you'd like to proceed!



In [ ]:
# Verify product creation
from utils import get_product
import json

product = get_product(product_id='PROD-200')
print(json.dumps(product, indent=2))

{
  "success": true,
  "product_id": "PROD-200",
  "details": "Product: Premium Gaming Headset\nProduct ID: PROD-200\nCategory: Audio\nPrice: $129.99\n\nDescription:\nProfessional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.\n\nSpecifications:\n  - bluetooth: 5.3\n  - driver_size: 50mm\n  - weight: 320g\n\nAvailability: Out of Stock\nStock Quantity: 0\n\n\nRating: 0 out of 5\n\nWarranty: 1 year manufacturer warranty\nReturn Policy: 30-day return policy\n",
  "product_data": {
    "category": "Audio",
    "created_date": "2026-02-27 05:28:51",
    "description": "Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.",
    "discontinued_date": "2026-02-27 13:16:33",
    "in_stock": false,
    "name": "Premium Gaming Headset",
    "price": 129.99,
    "product_id": "PROD-200",
    "rating": 0.0,
    "return_policy": "30-day return policy",
    "sale_end_date": "2025-03-31",
    "sale_price": 99.99,
    "specificat

In [32]:
# Admin Test 3: Update pricing with a sale
print("=" * 60)
print("Admin Test 3: Update Pricing (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Set a sale price of $99.99 for PROD-200 (regular price stays $129.99) "
    "until 2025-03-31"
)
print(f"\nResponse: {response}")

Admin Test 3: Update Pricing (ALLOWED)

Response: ✅ **Sale Price Successfully Set for PROD-200**

**Product:** Premium Gaming Headset

**Pricing Details:**
- **Regular Price:** $129.99 (unchanged)
- **Sale Price:** $99.99
- **Discount:** 23.1% off
- **Sale Period:** Until March 31, 2025

The product is now on sale! Customers will see the sale price of $99.99 when viewing this product, with the regular price of $129.99 displayed as the original price. The sale will automatically expire on 2025-03-31.



In [ ]:
# Verify product pricing update
from utils import get_product
import json

product = get_product(product_id='PROD-200')
print(json.dumps(product, indent=2))

DynamoDB Verification: PROD-200 (after pricing update)
  Product ID    : PROD-200
  Name          : Premium Gaming Headset
  Category      : Audio
  Price         : $129.99
  In Stock      : False
  Stock Qty     : 0.0
  Rating        : 0.0
  Warranty      : 1 year manufacturer warranty
  Return Policy : 30-day return policy
  Created Date  : 2026-02-27 05:28:51
  Updated Date  : 2026-02-27 13:16:33

  Description   : Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.

  Specifications:
    - bluetooth: 5.3
    - driver_size: 50mm
    - weight: 320g

Raw item (JSON):
{
  "category": "Audio",
  "created_date": "2026-02-27 05:28:51",
  "description": "Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.",
  "discontinued_date": "2026-02-27 13:16:33",
  "in_stock": false,
  "name": "Premium Gaming Headset",
  "price": 129.99,
  "product_id": "PROD-200",
  "rating": 0.0,
  "return_policy": "30-day return po

In [34]:
# Admin Test 4: Update inventory
print("=" * 60)
print("Admin Test 4: Update Inventory (ALLOWED)")
print("=" * 60)
response = admin_agent("Set the inventory for PROD-088 (4K webcam) to 100 units")
print(f"\nResponse: {response}")

Admin Test 4: Update Inventory (ALLOWED)

Response: ✅ **Inventory Successfully Updated for PROD-088**

**Product:** Webcam 4K HDR

**Inventory Details:**
- **Product ID:** PROD-088
- **New Stock Quantity:** 100 units
- **Status:** ✅ In Stock

The inventory for the 4K webcam has been updated to 100 units and is now marked as in stock.



In [ ]:
# Verify product inventory update
from utils import get_product
import json

product = get_product(product_id='PROD-088')
print(f"PROD-088 product update - {product['product_data']['stock_quantity']}")

{
  "success": true,
  "product_id": "PROD-088",
  "details": "Product: Webcam 4K HDR\nProduct ID: PROD-088\nCategory: Cameras\nPrice: $199.99\n\nDescription:\nProfessional 4K webcam with HDR, auto-focus, dual microphones, and privacy shutter.\n\nSpecifications:\n  - resolution: 4K @ 30fps, 1080p @ 60fps\n  - hdr: HDR support\n  - field_of_view: 90 degrees\n  - microphones: Dual omnidirectional\n  - features: ['Auto-focus', 'Auto light correction', 'Privacy shutter']\n\nAvailability: In Stock\nStock Quantity: 100\n\n\nRating: 4.7 out of 5\n\nWarranty: 2 years\nReturn Policy: 30-day return policy\n",
  "product_data": {
    "category": "Cameras",
    "description": "Professional 4K webcam with HDR, auto-focus, dual microphones, and privacy shutter.",
    "in_stock": true,
    "name": "Webcam 4K HDR",
    "price": 199.99,
    "product_id": "PROD-088",
    "rating": 4.7,
    "return_policy": "30-day return policy",
    "specifications": "{\"resolution\": \"4K @ 30fps, 1080p @ 60fps\", \"h

In [36]:
# Admin Test 5: Delete (discontinue) a product
print("=" * 60)
print("Admin Test 5: Delete Product (ALLOWED - soft delete)")
print("=" * 60)
response = admin_agent("Discontinue product PROD-200")
print(f"\nResponse: {response}")

Admin Test 5: Delete Product (ALLOWED - soft delete)

Response: ✅ **Product Successfully Discontinued**

**Product:** PROD-200 - Premium Gaming Headset

**Action Taken:** Soft Delete (Discontinued)

**Details:**
- The product has been marked as discontinued and removed from the active catalog
- The product record is **retained in the system** for order history and reference purposes
- Customers will no longer be able to purchase this product
- Any existing orders containing this product remain intact for historical records

**Note:** This is a soft delete, not a permanent removal. The product data is preserved in case you need to reference past orders or reactivate the product in the future.



In [ ]:
# Clean up admin agent
admin_agent.cleanup()
print("Admin agent cleaned up")

## Step 9: Access Control Boundary Test

Let's run a cross-role validation to confirm the RBAC boundary holds:
1. Admin creates a product → Customer can find it via search
2. Customer cannot modify the product the admin created

In [ ]:
# Create both agents
admin_agent = create_product_catalog_agent(region=REGION, user_session=admin_session)
customer_agent = create_product_catalog_agent(region=REGION, user_session=customer_session)

print(f"Admin tools:    {admin_agent.get_user_info()['tools_available']} tools")
print(f"Customer tools: {customer_agent.get_user_info()['tools_available']} tools")

In [ ]:
# Step A: Admin creates a new product
print("=" * 60)
print("Step A: Admin creates product PROD-300")
print("=" * 60)
response = admin_agent(
    "Create product PROD-300 called 'Wireless Charging Pad' in Accessories category "
    "for $39.99. Description: 'Fast wireless charger compatible with all Qi devices, "
    "supports up to 15W charging.' Specs: charging speed 15W, compatibility Qi, weight 120g. "
    "Stock: 200 units."
)
print(f"\nAdmin Response: {response}")

In [ ]:
# Step B: Customer can find the new product
print("=" * 60)
print("Step B: Customer searches for the new product")
print("=" * 60)
response = customer_agent("Do you have any wireless charging pads?")
print(f"\nCustomer Response: {response}")

In [ ]:
# Step C: Customer tries to modify the product (SHOULD FAIL)
print("=" * 60)
print("Step C: Customer tries to change the price (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Change the price of PROD-300 to $19.99")
print(f"\nCustomer Response: {response}")

In [ ]:
# Step D: Clean up - admin deletes the test product
print("=" * 60)
print("Step D: Admin cleans up test product")
print("=" * 60)
response = admin_agent("Delete product PROD-300")
print(f"\nAdmin Response: {response}")

In [ ]:
# Clean up both agents
admin_agent.cleanup()
customer_agent.cleanup()
print("Both agents cleaned up")

## Step 10: From Prototype to Production — AgentCore Identity

In this module, we simulated RBAC using a local `UserSession` class. In **Module 3** (Production Deployment), this maps to real authentication using **Amazon Bedrock AgentCore Identity**:

### Production Architecture

```
┌────────────┐     JWT Token      ┌─────────────────────┐
│  Web App   │ ──────────────────► │  AgentCore Runtime  │
│  (React)   │                     │                     │
└─────┬──────┘                     │  JWT Authorizer     │
      │                            │  validates token    │
      │ Login                      │                     │
      ▼                            │  Agent code reads   │
┌─────────────┐                    │  role from claims   │
│  Cognito    │                    │                     │
│  User Pool  │                    │  Tools filtered     │
│             │                    │  by role            │
│ Groups:     │                    └─────────────────────┘
│ - customer  │
│ - admin     │
└─────────────┘
```

### Key Mapping

| Prototype (Module 01) | Production (Module 03) |
|----------------------|------------------------|
| `UserSession` class | JWT token claims |
| `role` field | Cognito group membership |
| `user_id` field | JWT `sub` claim |
| `email` field | JWT `email` claim |
| Local tool filtering | Same logic, but role from verified JWT |

### Why This Pattern Works
1. **Defense in depth**: Tool filtering prevents the LLM from even knowing about restricted tools
2. **System prompt reinforcement**: Role-aware prompts add a second layer of guidance
3. **Verifiable**: You can inspect which tools an agent received at construction time
4. **Standard OAuth/JWT**: AgentCore Identity uses industry-standard authentication

## Summary

In this module, you built a **single Product Catalog Agent** with role-based access control that:

1. **Connects to an MCP server** with 11 tools (6 read + 5 write)
2. **Filters tools by role** — customers get 6 read tools, admins get all 11
3. **Uses role-aware system prompts** — different instructions per persona
4. **Validates access control** — customer cannot perform admin operations

### Key Takeaways
- **Start simple**: Validate one agent thoroughly before building multi-agent systems
- **RBAC via tool filtering**: Removing tools from the agent is a strong access control boundary
- **Persona testing**: Same agent code, different behaviors based on role
- **Production-ready pattern**: Local prototype maps directly to AgentCore Identity JWT auth

### Key Files

| File | Description |
|------|-------------|
| `mcp_servers/product_mcp_server.py` | MCP server with 11 product tools (6 read + 5 admin) |
| `agents/product_catalog_agent.py` | Agent with RBAC (UserSession, tool filtering, role prompts) |

### Next Steps

In **Module 2**, we'll evaluate this agent with a comprehensive test dataset to establish baseline metrics for quality, tool accuracy, and access control compliance.

In [ ]:
# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")